# 2026_PID_CRISTIAN_DAMIAN_FORTUNESKY_BARRIOS

## Actividad: Imagen obtenida mediante camara oscura

Este cuaderno resuelve un pipeline de mejora y restauracion para una imagen de camara oscura con degradaciones de bajo brillo, bajo contraste, rotacion, ruido y exceso de fondo no informativo.

El criterio metodologico sigue una secuencia de operaciones acotada:

1. recorte de region util;
2. rotacion para normalizar orientacion;
3. suavizado para atenuacion de ruido;
4. mejora local de contraste con CLAHE.


## Fundamento tecnico del enfoque

La imagen de camara oscura concentra informacion util en una zona pequena y de baja energia luminica. En este escenario, aplicar mejoras globales sobre todo el cuadro suele amplificar ruido en el fondo. Por eso se prioriza una estrategia por etapas:

- **Recorte**: reduce el dominio de procesamiento a la zona con senal util y evita que el fondo oscuro domine la estadistica de intensidad.
- **Rotacion**: corrige la orientacion del objeto para facilitar lectura visual y comparacion de resultados.
- **Suavizado mediana**: atenúa ruido impulsivo conservando bordes mejor que un promedio simple.
- **CLAHE en luminancia**: incrementa contraste local limitando sobre-amplificacion de ruido mediante `clipLimit`.

La combinacion busca maximizar visibilidad de la zona importante sin introducir artefactos agresivos.


In [ ]:
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 5)

## Carga de imagen y utilidades

La ruta de entrada definida para la actividad es `005/003/LAB/img/img-camara-oscura.jpg`. Se agrega una busqueda alternativa por nombre de archivo dentro del repositorio para mantener reproducibilidad si cambia la ubicacion exacta.


In [ ]:
def buscar_imagen_camara_oscura():
    candidatos = [
        Path('005/003/LAB/img/img-camara-oscura.jpg'),
        Path('img-camara-oscura.jpg'),
        Path('005 - TFI_1/img-camara-oscura.jpg')
    ]
    for p in candidatos:
        if p.exists():
            return p

    for p in Path('.').rglob('img-camara-oscura.jpg'):
        return p

    raise FileNotFoundError('No se encontro la imagen img-camara-oscura.jpg en el repositorio.')


def cargar_rgb(path):
    bgr = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if bgr is None:
        raise FileNotFoundError(f'No se pudo abrir: {path}')
    return cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)


def mostrar(imagen, titulo, cmap=None):
    plt.figure()
    if cmap is None:
        plt.imshow(imagen)
    else:
        plt.imshow(imagen, cmap=cmap)
    plt.title(titulo)
    plt.axis('off')
    plt.show()


def mostrar_histograma_gris(imagen_rgb, titulo='Histograma en escala de grises'):
    gris = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2GRAY)
    hist = cv2.calcHist([gris], [0], None, [256], [0, 256]).ravel()
    plt.figure(figsize=(8,3))
    plt.plot(hist)
    plt.title(titulo)
    plt.xlabel('Nivel de gris')
    plt.ylabel('Frecuencia')
    plt.show()

In [ ]:
ruta_imagen = buscar_imagen_camara_oscura()
imagen_original = cargar_rgb(ruta_imagen)

print(f'Imagen cargada desde: {ruta_imagen}')
print(f'Dimensiones: {imagen_original.shape[1]}x{imagen_original.shape[0]}')

mostrar(imagen_original, 'Imagen original - camara oscura')
mostrar_histograma_gris(imagen_original, 'Histograma original')

## Diagnostico inicial cuantitativo

Se usan dos indicadores sobre escala de grises:

- **Brillo medio**: aproxima nivel de iluminacion global.
- **Desvio estandar**: aproxima contraste global.

Valores bajos en ambos indicadores son consistentes con la degradacion observada en la imagen de entrada.


In [ ]:
gris_original = cv2.cvtColor(imagen_original, cv2.COLOR_RGB2GRAY)
brillo_medio_original = float(np.mean(gris_original))
contraste_original = float(np.std(gris_original))

print(f'Brillo medio original: {brillo_medio_original:.2f}')
print(f'Contraste (desvio estandar) original: {contraste_original:.2f}')

## Paso 1: Recorte de region util

Fundamento: el objeto de interes aparece como una isla de mayor intensidad sobre un fondo casi negro. Se segmenta una mascara en el canal V de HSV usando un umbral por percentil alto y se extrae la caja envolvente con margen.

Esta etapa reduce fondo inutil y evita que el histograma quede dominado por pixels oscuros del entorno.


In [ ]:
def recorte_region_util(imagen_rgb, percentil=98.5, margen=20):
    hsv = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2HSV)
    v = hsv[:, :, 2]

    umbral = np.percentile(v, percentil)
    mascara = (v >= umbral).astype(np.uint8) * 255

    kernel = np.ones((5, 5), np.uint8)
    mascara = cv2.morphologyEx(mascara, cv2.MORPH_CLOSE, kernel, iterations=2)
    mascara = cv2.GaussianBlur(mascara, (5, 5), 0)

    ys, xs = np.where(mascara > 0)
    if len(xs) == 0 or len(ys) == 0:
        return imagen_rgb.copy(), mascara

    x1, x2 = max(xs.min() - margen, 0), min(xs.max() + margen, imagen_rgb.shape[1])
    y1, y2 = max(ys.min() - margen, 0), min(ys.max() + margen, imagen_rgb.shape[0])

    recorte = imagen_rgb[y1:y2, x1:x2]
    return recorte, mascara


imagen_recortada, mascara_roi = recorte_region_util(imagen_original, percentil=98.5, margen=35)

mostrar(mascara_roi, 'Mascara de region util', cmap='gray')
mostrar(imagen_recortada, 'Paso 1 - Imagen recortada')

## Paso 2: Rotacion de alineacion

Fundamento: la orientacion inclinada reduce legibilidad de la zona util. Se estima el angulo principal de la mascara de alta intensidad con `minAreaRect` y se aplica una rotacion afín alrededor del centro de la imagen recortada.

La normalizacion angular facilita la interpretacion del contenido y estabiliza comparaciones posteriores.


In [ ]:
def estimar_angulo_principal(imagen_rgb):
    gris = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2GRAY)
    umbral = np.percentile(gris, 97)
    binaria = (gris >= umbral).astype(np.uint8) * 255

    contornos, _ = cv2.findContours(binaria, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contornos:
        return 0.0

    contorno = max(contornos, key=cv2.contourArea)
    rect = cv2.minAreaRect(contorno)
    angulo = rect[-1]

    if angulo < -45:
        angulo = 90 + angulo
    return float(angulo)


def rotar_imagen(imagen_rgb, angulo):
    h, w = imagen_rgb.shape[:2]
    centro = (w // 2, h // 2)
    matriz = cv2.getRotationMatrix2D(centro, angulo, 1.0)
    return cv2.warpAffine(imagen_rgb, matriz, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


angulo_estimado = estimar_angulo_principal(imagen_recortada)
imagen_rotada = rotar_imagen(imagen_recortada, -angulo_estimado)

print(f'Angulo estimado para correccion: {angulo_estimado:.2f} grados')
mostrar(imagen_rotada, 'Paso 2 - Imagen rotada/alineada')

## Paso 3: Suavizado para reduccion de ruido

Fundamento: en condiciones de baja iluminacion aparece ruido de sensor y compresion. Se usa filtro de mediana (`ksize=3`) por su capacidad de reducir ruido impulsivo sin borrar completamente los bordes de la region iluminada.


In [ ]:
imagen_suavizada = cv2.medianBlur(imagen_rotada, 3)
mostrar(imagen_suavizada, 'Paso 3 - Suavizado mediana')

## Paso 4: Mejora de contraste con CLAHE

Fundamento: la ecualizacion adaptativa por bloques aumenta contraste local en zonas de baja dinamica. Para evitar sobreamplificacion de ruido se aplica sobre canal L del espacio LAB con `clipLimit` moderado.


In [ ]:
def aplicar_clahe_lab(imagen_rgb, clip_limit=2.0, tile=(8,8)):
    lab = cv2.cvtColor(imagen_rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile)
    l2 = clahe.apply(l)

    lab2 = cv2.merge([l2, a, b])
    return cv2.cvtColor(lab2, cv2.COLOR_LAB2RGB)


imagen_final = aplicar_clahe_lab(imagen_suavizada, clip_limit=2.2, tile=(8,8))
mostrar(imagen_final, 'Paso 4 - Imagen final con CLAHE')
mostrar_histograma_gris(imagen_final, 'Histograma final')

## Comparacion antes/despues

La comparacion sintetiza el efecto acumulado del pipeline completo.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].imshow(imagen_original)
ax[0].set_title('Original')
ax[0].axis('off')

ax[1].imshow(imagen_final)
ax[1].set_title('Resultado final')
ax[1].axis('off')

plt.tight_layout()
plt.show()

## Evaluacion cuantitativa de mejora

Se reportan los mismos indicadores usados en el diagnostico inicial para sostener la decision tecnica.


In [ ]:
gris_final = cv2.cvtColor(imagen_final, cv2.COLOR_RGB2GRAY)
brillo_medio_final = float(np.mean(gris_final))
contraste_final = float(np.std(gris_final))

print(f'Brillo medio final: {brillo_medio_final:.2f}')
print(f'Contraste final: {contraste_final:.2f}')
print(f'Delta brillo: {brillo_medio_final - brillo_medio_original:.2f}')
print(f'Delta contraste: {contraste_final - contraste_original:.2f}')

## Conclusiones tecnicas

1. El recorte de region util disminuyo el impacto estadistico del fondo oscuro y concentro el procesamiento en la zona informativa.
2. La correccion de orientacion mejoro la lectura espacial del objeto principal.
3. El suavizado mediana redujo ruido de baja iluminacion con perdida limitada de bordes.
4. CLAHE sobre luminancia incremento contraste local y visibilidad de detalles internos.
5. Como limite del metodo, la informacion ausente por subexposicion extrema no puede recuperarse completamente.


## Guardado de salida

El resultado final se guarda en `salidas_tfi/camara_oscura_final.png` para integracion posterior con el TFI completo.


In [ ]:
carpeta_salida = Path('salidas_tfi')
carpeta_salida.mkdir(parents=True, exist_ok=True)

ruta_salida = carpeta_salida / 'camara_oscura_final.png'
cv2.imwrite(str(ruta_salida), cv2.cvtColor(imagen_final, cv2.COLOR_RGB2BGR))
print(f'Salida guardada en: {ruta_salida}')